<a href="https://colab.research.google.com/github/AnithaN21/AI-12-days-Bootcamp/blob/main/Day6_FN_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic

In [2]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [4]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'''
Extract a Resume JSON from this text.
Return ONLY JSON, no markdown.

{raw_text}
''',

                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = f'''
Fix this JSON to match schema.

Errors:
{e}

Original:
{resp.text}
'''

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,

                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                }
            )

            return Resume.model_validate_json(resp.text)

In [5]:
from google.colab import files
uploaded = files.upload()

Saving sample_resumes.txt to sample_resumes.txt


In [6]:
# Load 5 sample résumés

with open('sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []

for i, r in enumerate(resumes):

    try:
        parsed = extract_resume(r)

        results.append(parsed)

        print(f'[{i+1}] {parsed.name} — {len(parsed.skills)} skills')

    except Exception as e:

        errors.append((i, e))

        print(f'[{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
[1] Ravi Kumar — 6 skills
[2] Sneha Reddy — 6 skills
[3] Arun Pillai — 8 skills
[4] Priya Nair — 5 skills
[5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


In [7]:
# Empty string
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"No name provided","email":"no_email@example.com","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: {"name":"N/A","email":"n/a@example.com","phone":null,"education":[{"degree":"N/A","institution":"N/A University","year":2000}],"skills":["N/A"],"projects":[],"experience_years":0.0}


In [8]:
!pip install -q google-genai beautifulsoup4 requests pydantic

In [9]:
from google import genai
from pydantic import BaseModel
from typing import List, Optional
import requests
from bs4 import BeautifulSoup
import json
import pathlib
import os

In [10]:
import getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Enter API Key: ')

In [11]:
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

In [12]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [13]:
def fetch_jd(url, max_chars=6000):

    try:
        response = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=10
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # remove scripts/styles
        for tag in soup(['script', 'style']):
            tag.decompose()

        text = soup.get_text(
            separator='\n',
            strip=True
        )

        return text[:max_chars]

    except Exception as e:
        print("Scraping failed:", e)
        return None

In [14]:
url = url = "https://www.geeksforgeeks.org/jobs/"

text = fetch_jd(url)

print(text[:1000])

GFG Get Hired
Courses
Tutorials
Interview Prep
Get Hired by
GfG
Prepare for
top tech roles
with
structured learning
,
hands-on practice
, and real
job opportunities
, all in one career platform.
150+
Organizations
200+
Job Openings
6 LPA
Average Package
Apply Filters
Reset
Filters
Experience
Select experience
< 1 year
1 to 3 years
3 to 5 years
More than 5 years
Location Type
Select location type
Onsite
Hybrid
Remote
Location
Job Type
Select job type
Technology & Engineering
Business, Finance & Management
Sales, Marketing & Customer Relations
Education, Research & Social Services
HR & Operations
Manufacturing & Logistics
View Applied Jobs
Filter Jobs
×
Experience
Select experience
< 1 year
1 to 3 years
3 to 5 years
More than 5 years
Location Type
Select location type
Onsite
Hybrid
Remote
Location
Job Type
Select job type
Technology & Engineering
Business, Finance & Management
Sales, Marketing & Customer Relations
Education, Research & Social Services
HR & Operations
Manufacturing & Logi

In [15]:
def normalize_jd(text):

    response = client.models.generate_content(

        model='gemini-2.5-flash',

        contents=f"""
        Extract a Job Description JSON from this text.

        Return ONLY valid JSON.

        Text:
        {text}
        """,

        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema()
        }
    )

    return JD.model_validate_json(response.text)

In [16]:
jd = normalize_jd(text)

print(jd.model_dump_json(indent=2))

{
  "company": "GfG",
  "role": "Software Engineer",
  "must_have_skills": [
    "Programming Languages",
    "Data Structures and Algorithms",
    "Web Technology",
    "AI, ML & Data Science",
    "DevOps",
    "CS Core Subjects",
    "System Design"
  ],
  "nice_to_have_skills": [],
  "min_cgpa": null,
  "locations": [
    "Noida, Uttar Pradesh"
  ],
  "package_lpa": 6.0
}


In [18]:
URLS = [
    "https://www.geeksforgeeks.org/jobs/",
    "https://careers.ibm.com/",
    "https://jobs.careers.microsoft.com/"
]

In [19]:
jds = []

for url in URLS:

    print(f"Processing: {url}")

    text = fetch_jd(url)

    if text is None:
        print("Scraping failed")
        continue

    try:

        jd = normalize_jd(text)

        jds.append(jd)

        print(f"✓ {jd.company} - {jd.role}")

    except Exception as e:

        print("Failed:", e)

print(f"\nProcessed {len(jds)} JDs")

Processing: https://www.geeksforgeeks.org/jobs/
✓ GeeksforGeeks - Tech Roles
Processing: https://careers.ibm.com/
✓  - 
Processing: https://jobs.careers.microsoft.com/
✓ Microsoft - 

Processed 3 JDs


In [20]:
import pathlib

OUT = pathlib.Path("data/jds.jsonl")

OUT.parent.mkdir(exist_ok=True)

with open(OUT, "w") as f:

    for jd in jds:

        f.write(jd.model_dump_json() + "\n")

print("JSONL file saved successfully")

JSONL file saved successfully


In [22]:
import json

with open(OUT) as f:

    for line in f:

        d = json.loads(line)

        print(
            d["company"],
            "|",
            d["role"]
        )

GeeksforGeeks | Tech Roles
 | 
Microsoft | 
